In [1]:
# CELDA 1 — Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# CELDA 2 — Imports y configuración de rutas
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Semilla aleatoria global → garantiza reproducibilidad (siempre mismo resultado)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Rutas del proyecto (deben coincidir con las carpetas que creaste en Drive)
PROJECT_PATH = '/content/drive/MyDrive/HortifrutCostosImport'
RAW_PATH       = f'{PROJECT_PATH}/data/raw'
PROCESSED_PATH = f'{PROJECT_PATH}/data/processed'
REF_PATH       = f'{PROJECT_PATH}/referencias'

print(f"Proyecto: {PROJECT_PATH}")
print(f"Semilla aleatoria: {RANDOM_SEED}")

Proyecto: /content/drive/MyDrive/HortifrutCostosImport
Semilla aleatoria: 42


In [3]:
# CELDA 3 — Sanity check: verificar carpetas y archivos
print("=== Carpetas ===")
for p in [PROJECT_PATH, RAW_PATH, PROCESSED_PATH, REF_PATH]:
    status = "✓" if os.path.exists(p) else "✗ FALTA"
    print(f"  {status}  {p}")

print("\n=== Archivos crudos esperados en data/raw ===")
expected_raw = [
    'bd_expense_report_importaciones_201X.csv',
    'report_importaciones_2019-2020.csv',
    'report_importaciones_2021.csv',
    'report_importaciones_2022.csv',
    'report_importaciones_2023.csv',
    'report_importaciones_2024.csv',
    'report_importaciones_2025.csv',
    'report_importaciones_2026.csv',
    'Diccionario Variables.xlsx',
]
for f in expected_raw:
    status = "✓" if os.path.exists(f'{RAW_PATH}/{f}') else "✗ FALTA"
    print(f"  {status}  {f}")

print("\n=== Archivos de referencia que ya generamos ===")
for f in ['mapping_status_schemas.xlsx', 'conceptos_canonicos.xlsx']:
    status = "✓" if os.path.exists(f'{REF_PATH}/{f}') else "✗ FALTA"
    print(f"  {status}  {f}")

=== Carpetas ===
  ✓  /content/drive/MyDrive/HortifrutCostosImport
  ✓  /content/drive/MyDrive/HortifrutCostosImport/data/raw
  ✓  /content/drive/MyDrive/HortifrutCostosImport/data/processed
  ✓  /content/drive/MyDrive/HortifrutCostosImport/referencias

=== Archivos crudos esperados en data/raw ===
  ✓  bd_expense_report_importaciones_201X.csv
  ✓  report_importaciones_2019-2020.csv
  ✓  report_importaciones_2021.csv
  ✓  report_importaciones_2022.csv
  ✓  report_importaciones_2023.csv
  ✓  report_importaciones_2024.csv
  ✓  report_importaciones_2025.csv
  ✓  report_importaciones_2026.csv
  ✓  Diccionario Variables.xlsx

=== Archivos de referencia que ya generamos ===
  ✓  mapping_status_schemas.xlsx
  ✓  conceptos_canonicos.xlsx


In [4]:
# CELDA 4 — Cargar archivos de referencia (mapping de schemas y conceptos canónicos)
mapping = pd.read_excel(
    f'{REF_PATH}/mapping_status_schemas.xlsx',
    sheet_name='1. Mapeo Variables'
)
disponibilidad = pd.read_excel(
    f'{REF_PATH}/mapping_status_schemas.xlsx',
    sheet_name='2. Disponibilidad al Predecir'
)
conceptos_map = pd.read_excel(
    f'{REF_PATH}/conceptos_canonicos.xlsx',
    sheet_name='1. Mapeo Conceptos'
)

print(f"Variables del diccionario:  {len(mapping)}")
print(f"Conceptos a consolidar:     {len(conceptos_map)}")
print(f"Reglas de disponibilidad:   {len(disponibilidad)}")

# Construimos un diccionario Python {concepto_original: concepto_canonico}
# Esto lo vamos a usar más adelante para consolidar los 81 → 13.
CONCEPTO_TO_CANON = dict(zip(
    conceptos_map['Concepto Original'],
    conceptos_map['Concepto Canónico Propuesto']
))

print(f"\nEjemplos del mapeo canónico:")
for k, v in list(CONCEPTO_TO_CANON.items())[:5]:
    print(f"  '{k}'  →  {v}")

Variables del diccionario:  39
Conceptos a consolidar:     81
Reglas de disponibilidad:   39

Ejemplos del mapeo canónico:
  'Descarga'  →  DESCARGA
  'Transporte Lima - Trujillo'  →  TRANSPORTE_T2_LIMA_FUNDO
  'Derechos'  →  DERECHOS_IMPUESTOS
  'Transporte Callao - Lima'  →  TRANSPORTE_T1_CALLAO_LIMA
  'Agenciamiento de Aduana'  →  AGENCIAMIENTO_ADUANA


In [5]:
# CELDA 5 — Cargar EXPENSE Report
expense = pd.read_csv(
    f'{RAW_PATH}/bd_expense_report_importaciones_201X.csv',
    encoding='utf-8-sig',
    low_memory=False
)
expense = expense.dropna(how='all')  # sacar filas 100% vacías

# Limpieza ligera de la columna Campaña para que sea numérica
expense['Campaña'] = pd.to_numeric(expense['Campaña'], errors='coerce').astype('Int64')

print(f"Shape: {expense.shape[0]:,} filas × {expense.shape[1]} columnas")
print(f"Operaciones únicas: {expense['Nro. Ope.'].nunique():,}")
print(f"\nDistribución por campaña:")
print(expense['Campaña'].value_counts().sort_index().to_string())
print(f"\nMonedas:")
print(expense['Moneda'].value_counts().to_string())

print("\nPrimeras 3 filas:")
expense.head(3)

Shape: 14,879 filas × 39 columnas
Operaciones únicas: 1,278

Distribución por campaña:
Campaña
2019     174
2020     999
2021    3412
2022    3477
2023    1546
2024    2892
2025    2329
2026      50

Monedas:
Moneda
USD    10237
PEN     4474
EUR      168

Primeras 3 filas:


,Campaña,Nro. Ope.,Versión Nro. Ope.,OC,Importador/SOC,Área Usuaria,POL,Proveedor Principal,Producto,FACTURA,...,Count N° Import,Agrupador,Clasificación,Market,Country,Familia de Producto,Gerencia,Comprador,OC2,Unnamed: 38
0,2019,19-061HFP-184,1.0,4700001432,HORTIFRUT-PERU S.A.C.,Operaciones Agrícolas,VALENCIA,"AGROQUÍMICA CODIAGRO, S.L.",FITOKONTROL,955,...,#¡REF!,Base,AGENCIAMIENTO,EUROPE,SPAIN,AGROQUIMICOS,#N/D,#N/D,4700001432,NaN
1,2019,19-061HFP-184,1.0,4700001432,HORTIFRUT-PERU S.A.C.,Operaciones Agrícolas,VALENCIA,"AGROQUÍMICA CODIAGRO, S.L.",FITOKONTROL,955,...,0,Base,FLETE NACIONAL,EUROPE,SPAIN,AGROQUIMICOS,#N/D,#N/D,4700001432,NaN
2,2019,19-061HFP-184,1.0,4700001432,HORTIFRUT-PERU S.A.C.,Operaciones Agrícolas,VALENCIA,"AGROQUÍMICA CODIAGRO, S.L.",FITOKONTROL,955,...,0,Base,FLETE INTERNACIONAL,EUROPE,SPAIN,AGROQUIMICOS,#N/D,#N/D,4700001432,NaN


In [6]:
# CELDA 6 — Cargar y unificar los 7 Status Reports

# Paso 1: construir las reglas de renombrado a partir del Excel de mapping.
# Para cada año, el resultado es un dict {"nombre original en archivo": "nombre canónico"}.
def construir_renames(mapping_df, year_label):
    col_name = f'STATUS {year_label}'
    renames = {}
    for _, row in mapping_df.iterrows():
        original = row[col_name]
        canonico = row['Variable Canónica']
        # Saltar: vacíos, guiones, y casos compuestos (los manejamos aparte)
        if pd.isna(original) or str(original).strip() in ('—', '') or '+' in str(original):
            continue
        renames[str(original).strip()] = canonico
    return renames

RENAMES = {y: construir_renames(mapping, y) for y in
           ['2019-2020', '2021', '2022', '2023', '2024', '2025', '2026']}

# Paso 2: archivos a cargar
STATUS_FILES = {
    '2019-2020': 'report_importaciones_2019-2020.csv',
    '2021':      'report_importaciones_2021.csv',
    '2022':      'report_importaciones_2022.csv',
    '2023':      'report_importaciones_2023.csv',
    '2024':      'report_importaciones_2024.csv',
    '2025':      'report_importaciones_2025.csv',
    '2026':      'report_importaciones_2026.csv',
}

# Paso 3: cargar cada archivo con sus renames
status_dfs = []
for year_label, filename in STATUS_FILES.items():
    df_y = pd.read_csv(f'{RAW_PATH}/{filename}', encoding='utf-8-sig', low_memory=False)
    df_y = df_y.dropna(how='all')               # importante para 2021 (tiene 1M de padding vacío)
    df_y.columns = [c.strip() for c in df_y.columns]  # limpiar espacios en nombres
    df_y = df_y.rename(columns=RENAMES[year_label])

    # Caso especial: 'Modalidad (MODE Y TYPE)' = concatenación de MODE + TYPE
    if 'MODE' in df_y.columns and 'TYPE' in df_y.columns:
        df_y['Modalidad (MODE Y TYPE)'] = (
            df_y['MODE'].fillna('').astype(str) + ' / ' +
            df_y['TYPE'].fillna('').astype(str)
        )

    df_y['_anio_archivo_status'] = year_label  # tag de trazabilidad
    status_dfs.append(df_y)
    print(f"  {year_label}: {df_y.shape[0]:>4} filas cargadas")

# Paso 4: concatenar los 7 dataframes
status = pd.concat(status_dfs, ignore_index=True, sort=False)
print(f"\nSTATUS unificado: {status.shape[0]:,} filas × {status.shape[1]} columnas")
print(f"Operaciones únicas en STATUS: {status['Nro. Ope.'].nunique():,}")

  2019-2020:  177 filas cargadas
  2021:  595 filas cargadas
  2022:  599 filas cargadas
  2023:  302 filas cargadas
  2024:  636 filas cargadas
  2025:  629 filas cargadas
  2026:  150 filas cargadas

STATUS unificado: 3,088 filas × 159 columnas
Operaciones únicas en STATUS: 1,431


In [7]:
# CELDA 7 — Agregar STATUS a nivel operación y mergear con EXPENSE

# Limpieza de columnas numéricas (vienen como string con comas y espacios)
for col in ['Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores', 'Peso Bruto (kg)']:
    if col in status.columns:
        status[col] = pd.to_numeric(
            status[col].astype(str)
                       .str.replace(',', '')
                       .str.strip()
                       .replace({'NA': np.nan, 'nan': np.nan, '': np.nan, '-': np.nan}),
            errors='coerce'
        )

# Conversión de fechas (formato peruano: día/mes/año)
for col in ['Fecha de Llegada (ETA)', 'Fecha de Despacho']:
    if col in status.columns:
        status[col] = pd.to_datetime(status[col], errors='coerce', dayfirst=True)

# Solo nos quedamos con features que aportan info nueva (no duplican EXPENSE)
features_a_traer = [
    'Nro. Ope.', 'Peso Bruto (kg)', 'Cantidad de Bultos (BULKS)',
    'Cantidad de Contenedores', 'Tipo de Contenedor', 'Modalidad (MODE Y TYPE)',
    'Incoterm', 'País de Origen (POL)', 'POD (Puerto Destino)',
    'Fecha de Llegada (ETA)', 'Fecha de Despacho',
    'Delivery type', 'Final delivery', 'Proyecto'
]
cols_existentes = [c for c in features_a_traer if c in status.columns]
status_subset = status[cols_existentes].copy()

# Agregación: sumar numéricas, tomar primer no-nulo para categóricas
num_cols = [c for c in ['Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores', 'Peso Bruto (kg)']
            if c in status_subset.columns]
cat_cols = [c for c in status_subset.columns if c not in (['Nro. Ope.'] + num_cols)]

status_num = status_subset.groupby('Nro. Ope.', as_index=False)[num_cols].sum(min_count=1)
status_cat = status_subset.groupby('Nro. Ope.', as_index=False)[cat_cols].first()
status_op  = status_num.merge(status_cat, on='Nro. Ope.')

print(f"STATUS a nivel operación: {len(status_op):,} operaciones únicas")

# Merge con EXPENSE (left join: conservamos todas las facturas)
df = expense.merge(status_op, on='Nro. Ope.', how='left')
print(f"Dataset mergeado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Verificar calidad del join
todas_nan = df[['Cantidad de Bultos (BULKS)', 'Cantidad de Contenedores', 'Incoterm']].isna().all(axis=1)
print(f"Filas sin features de status: {todas_nan.sum():,} ({todas_nan.sum()/len(df)*100:.1f}%)")

STATUS a nivel operación: 1,431 operaciones únicas
Dataset mergeado: 14,879 filas × 52 columnas
Filas sin features de status: 326 (2.2%)


In [8]:
# CELDA 8 — Consolidar conceptos a 13 canónicos y descartar 2019

# Aplicar el mapeo Concepto Original → Concepto Canónico (81 → 13 grupos)
df['Concepto Canónico'] = df['Concepto'].map(CONCEPTO_TO_CANON)

# Verificar que ningún concepto quedó sin mapear (debería ser 0)
sin_mapear = df[df['Concepto Canónico'].isna()]['Concepto'].dropna().unique()
print(f"Conceptos sin mapeo canónico: {len(sin_mapear)}")
if len(sin_mapear) > 0:
    print(f"  Ejemplos: {list(sin_mapear[:5])}")
else:
    print("  ✓ Todos los conceptos están mapeados")

print(f"\nDistribución por Concepto Canónico:")
print(df['Concepto Canónico'].value_counts().to_string())

# === Descarte de 2019 ===
# DECISIÓN DE DISEÑO (sesión 2026-05-15):
# Descartamos las 174 filas de 2019 porque NO existe un Status Report puro de 2019.
# El archivo "2019-2020" solo cubre operaciones del 2020 (formato YY-NNN-HPER).
# Las operaciones de 2019 tienen formato legacy (YY-NNNHFP-NNN) y no matchean.
# Entrenar con esas filas sería entrenar con features de despacho 100% nulas.
print(f"\nFilas antes de descartar 2019: {len(df):,}")
df = df[df['Campaña'] != 2019].reset_index(drop=True)
print(f"Filas después de descartar 2019: {len(df):,}")
print(f"Operaciones únicas finales: {df['Nro. Ope.'].nunique():,}")

Conceptos sin mapeo canónico: 0
  ✓ Todos los conceptos están mapeados

Distribución por Concepto Canónico:
Concepto Canónico
DERECHOS_IMPUESTOS           2173
INSPECCION_VERIFICACION      1898
DESCARGA                     1763
HANDLING_PUERTO              1599
TRANSPORTE_T2_LIMA_FUNDO     1539
FITOSANITARIOS_SENASA        1132
AGENCIAMIENTO_ADUANA         1072
FLETE_INTERNACIONAL          1052
TRANSPORTE_T1_CALLAO_LIMA     867
SEGUROS                       645
SERVICIOS_LOGISTICOS          591
SOBRESTADIA_ALMACENAJE        476
OTROS                          72

Filas antes de descartar 2019: 14,879
Filas después de descartar 2019: 14,705
Operaciones únicas finales: 1,252


In [9]:
# CELDA 9 — Guardar el dataset unificado

os.makedirs(PROCESSED_PATH, exist_ok=True)
output_path = f'{PROCESSED_PATH}/dataset_unificado.parquet'
df.to_parquet(output_path, index=False)

print(f"✓ Guardado en: {output_path}")
print(f"  Tamaño:   {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"  Filas:    {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"\nColumnas finales del dataset:")
for c in df.columns:
    print(f"  • {c}")

✓ Guardado en: /content/drive/MyDrive/HortifrutCostosImport/data/processed/dataset_unificado.parquet
  Tamaño:   654.4 KB
  Filas:    14,705
  Columnas: 53

Columnas finales del dataset:
  • Campaña
  • Nro. Ope.
  • Versión Nro. Ope.
  • OC
  • Importador/SOC
  • Área Usuaria
  • POL
  • Proveedor Principal
  • Producto
  • FACTURA
  • BL / AWB
  • AGENCIA DE ADUANA
  • ACREEDOR
  • Fecha de Emisión Doc
  • Nro Liquid.
  • Nro. de Doc.
  • Tipo de Doc.
  • Proveedor
  • Concepto
  • Moneda
  • Provisión
  • Monto Final
  • Igv
  • Importe Total
  • Fecha de Envío Provisión
  • Fecha de Envío Liquidación
  • Atención
  • Monto Total USD
  • Count OC
  • Count N° Import
  • Agrupador
  • Clasificación
  • Market
  • Country
  • Familia de Producto
  • Gerencia
  • Comprador
  • OC2
  • Unnamed: 38
  • Cantidad de Bultos (BULKS)
  • Cantidad de Contenedores
  • Peso Bruto (kg)
  • Tipo de Contenedor
  • Modalidad (MODE Y TYPE)
  • Incoterm
  • País de Origen (POL)
  • POD (Puerto Destino